Vanessa Wafa Wehbe

1. Download All Data (CSVs & Images)

In [1]:
# Download text metadata
!wget -q https://uni-bonn.sciebo.de/s/r7iYfS2QTDrA4CD/download/labels.csv -O labels.csv
!wget -q https://uni-bonn.sciebo.de/s/tjc7eF5CsynKwrG/download/train_labels.csv -O train_labels.csv
!wget -q https://uni-bonn.sciebo.de/s/KFyAPTgjqMPWiBA/download/test_labels.csv -O test_labels.csv
!wget -q https://uni-bonn.sciebo.de/s/f7yaDakFNS5C2n7/download/metadata_clean.csv -O metadata_clean.csv

# Download and extract the image ZIP file
!wget -q https://uni-bonn.sciebo.de/s/KrMiTk2X7sgBCwK/download/MIMIC-CXR-png.zip -O MIMIC-CXR-png.zip
!unzip -q -o MIMIC-CXR-png.zip -d /content/images

2. Setup & Imports

In [2]:
import os
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoProcessor, AutoModel
from peft import get_peft_model, LoraConfig
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


3. Data Merging

In [3]:
train_labels = pd.read_csv('train_labels.csv')
val_labels = pd.read_csv('test_labels.csv')
metadata = pd.read_csv('metadata_clean.csv')

# Merge labels with metadata based on image ID
train_df = train_labels.merge(metadata, on='dicom_id', how='left')
val_df = val_labels.merge(metadata, on='dicom_id', how='left')

train_df = train_df.dropna(subset=['mask_path']).reset_index(drop=True)
val_df = val_df.dropna(subset=['mask_path']).reset_index(drop=True)

# The root directory containing the nested 'segmentation' folders
IMAGE_DIR = "/content/images/MIMIC-CXR-png"

4. MIMIC-CXR Dataset Class

In [4]:
TABULAR_FEATURE_SIZE = 1 # We are using "View Position" (PA vs AP)

class MIMICCXR_MultimodalDataset(Dataset):
    def __init__(self, df, processor=None, is_medclip=False):
        self.df = df
        self.processor = processor
        self.is_medclip = is_medclip
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Load Image using the accurate mask_path
        relative_path = str(row['mask_path'])
        img_path = os.path.join(IMAGE_DIR, relative_path)
        image = Image.open(img_path).convert('RGB')

        if self.is_medclip and self.processor:
            image_tensor = self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        else:
            image_tensor = self.transform(image)

        # 2. Load Tabular Metadata (View Position)
        view = str(row['ViewCodeSequence_CodeMeaning']).lower()
        is_pa = 1.0 if 'postero-anterior' in view else 0.0
        tabular_tensor = torch.tensor([is_pa], dtype=torch.float32)

        # 3. Load Label
        lbl = row['pathology']
        # If the label is parsed as a string representation of a list, clean it
        if isinstance(lbl, str) and '[' in lbl:
            lbl = eval(lbl.replace(' ', ','))
        if isinstance(lbl, (list, np.ndarray, tuple)):
            lbl = lbl[0] # Grab the target pathology (e.g., Pneumonia)

        label = torch.tensor(float(lbl), dtype=torch.float32)

        return image_tensor, tabular_tensor, label

processor = AutoProcessor.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
train_loader_scratch = DataLoader(MIMICCXR_MultimodalDataset(train_df, is_medclip=False), batch_size=32, shuffle=True)
val_loader_scratch = DataLoader(MIMICCXR_MultimodalDataset(val_df, is_medclip=False), batch_size=32, shuffle=False)
train_loader_clip = DataLoader(MIMICCXR_MultimodalDataset(train_df, processor=processor, is_medclip=True), batch_size=32, shuffle=True)
val_loader_clip = DataLoader(MIMICCXR_MultimodalDataset(val_df, processor=processor, is_medclip=True), batch_size=32, shuffle=False)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

5. The 5 Models

In [5]:
class ResNetConcatModel(nn.Module):
    def __init__(self, num_tabular_features):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.vision_backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Sequential(nn.Linear(resnet.fc.in_features + num_tabular_features, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, image, tabular):
        img_features = self.vision_backbone(image).squeeze(-1).squeeze(-1)
        return self.classifier(torch.cat([img_features, tabular], dim=1))

class ResNetCrossAttentionModel(nn.Module):
    def __init__(self, num_tabular_features, embed_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.vision_backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.img_proj = nn.Linear(resnet.fc.in_features, embed_dim)
        self.tab_proj = nn.Linear(num_tabular_features, embed_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)
        self.classifier = nn.Sequential(nn.Linear(embed_dim, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, image, tabular):
        spatial_features = self.vision_backbone(image)
        B, C, H, W = spatial_features.shape
        kv = self.img_proj(spatial_features.view(B, C, -1).permute(0, 2, 1))
        q = self.tab_proj(tabular).unsqueeze(1)
        attn_output, _ = self.cross_attn(query=q, key=kv, value=kv)
        return self.classifier(attn_output.squeeze(1))

class MedCLIPLinearProbe(nn.Module):
    def __init__(self, vision_model, num_tabular_features):
        super().__init__()
        self.vision_model = vision_model
        for param in self.vision_model.parameters(): param.requires_grad = False
        self.classifier = nn.Linear(512 + num_tabular_features, 1)

    def forward(self, image, tabular):
        img_features = self.vision_model.get_image_features(pixel_values=image)
        if hasattr(img_features, 'pooler_output'): img_features = img_features.pooler_output
        return self.classifier(torch.cat([img_features, tabular], dim=1))

class MedCLIPPartialFT(nn.Module):
    def __init__(self, vision_model, num_tabular_features):
        super().__init__()
        self.vision_model = vision_model
        for param in self.vision_model.parameters(): param.requires_grad = False
        self.classifier = nn.Sequential(nn.Linear(512 + num_tabular_features, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, image, tabular):
        img_features = self.vision_model.get_image_features(pixel_values=image)
        if hasattr(img_features, 'pooler_output'): img_features = img_features.pooler_output
        return self.classifier(torch.cat([img_features, tabular], dim=1))

class MedCLIPLoRA(nn.Module):
    def __init__(self, vision_model, num_tabular_features):
        super().__init__()
        for param in vision_model.parameters(): param.requires_grad = False
        self.vision_model = get_peft_model(vision_model, LoraConfig(r=16, lora_alpha=16, target_modules=["q_proj", "v_proj"]))
        self.classifier = nn.Sequential(nn.Linear(512 + num_tabular_features, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, image, tabular):
        img_features = self.vision_model.get_image_features(pixel_values=image)
        if hasattr(img_features, 'pooler_output'): img_features = img_features.pooler_output
        return self.classifier(torch.cat([img_features, tabular], dim=1))

6. Training & Execution

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
import time
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn

# Define the ECE function
def calculate_ece(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0., 1., n_bins + 1)
    bin_indices = np.digitize(y_prob, bin_edges) - 1
    ece = 0.0
    for i in range(n_bins):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            diff = np.abs(np.mean(y_prob[mask]) - np.mean(y_true[mask]))
            weight = np.sum(mask) / len(y_true)
            ece += diff * weight
    return ece

def train_and_evaluate(model, model_name, train_loader, val_loader, epochs=10):
    print(f"\n--- Training {model_name} ---")
    model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()

    # Track the best metrics
    best_auc, best_acc, best_f1, best_prec, best_rec = 0, 0, 0, 0, 0
    best_ece = 1.0

    for epoch in range(epochs):
        model.train()
        for images, tabular, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            optimizer.zero_grad()
            loss = criterion(model(images.to(device), tabular.to(device)).squeeze(), labels.to(device))
            loss.backward()
            optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for images, tabular, labels in val_loader:
                outputs = model(images.to(device), tabular.to(device))
                all_probs.extend(torch.sigmoid(outputs.squeeze()).cpu().numpy())
                all_labels.extend(labels.numpy())

        all_preds = (np.array(all_probs) > 0.5).astype(int)
        y_true_np = np.array(all_labels)
        y_prob_np = np.array(all_probs)

        try:
            val_auc = roc_auc_score(y_true_np, y_prob_np)
        except ValueError:
            val_auc = 0.0

        val_acc = accuracy_score(y_true_np, all_preds)
        val_f1 = f1_score(y_true_np, all_preds, zero_division=0)

        # Calculate the new metrics
        val_prec = precision_score(y_true_np, all_preds, zero_division=0)
        val_rec = recall_score(y_true_np, all_preds, zero_division=0)
        val_ece = calculate_ece(y_true_np, y_prob_np)

        # Update best scores
        best_auc = max(best_auc, val_auc)
        best_acc = max(best_acc, val_acc)
        best_f1 = max(best_f1, val_f1)
        best_prec = max(best_prec, val_prec)
        best_rec = max(best_rec, val_rec)
        best_ece = min(best_ece, val_ece) # Added ECE

        print(f"Epoch {epoch+1} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | ECE: {val_ece:.4f}")

    training_time = time.time() - start_time
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else 0

    return {
        'Model': model_name,
        'Precision': round(best_prec, 4),
        'Recall': round(best_rec, 4),
        'AUC': round(best_auc, 4),
        'F1': round(best_f1, 4),
        'ECE': round(best_ece, 4),
        'Time (s)': round(training_time, 2),
        'GPU Mem (MB)': round(gpu_mem, 2),
        'Params': f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    }


results = []
EPOCHS = 10

# Train Scratch Models
results.append(train_and_evaluate(ResNetConcatModel(TABULAR_FEATURE_SIZE), "ResNet Concat (Scratch)", train_loader_scratch, val_loader_scratch, EPOCHS))
results.append(train_and_evaluate(ResNetCrossAttentionModel(TABULAR_FEATURE_SIZE), "ResNet Cross-Attn (Scratch)", train_loader_scratch, val_loader_scratch, EPOCHS))

# Train Foundation Models
base_medclip = AutoModel.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
results.append(train_and_evaluate(MedCLIPLinearProbe(base_medclip, TABULAR_FEATURE_SIZE), "MedCLIP Linear Probe", train_loader_clip, val_loader_clip, EPOCHS))
results.append(train_and_evaluate(MedCLIPPartialFT(base_medclip, TABULAR_FEATURE_SIZE), "MedCLIP Partial FT", train_loader_clip, val_loader_clip, EPOCHS))

# Reload fresh base for LoRA to avoid PEFT wrapper conflicts
fresh_base = AutoModel.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
results.append(train_and_evaluate(MedCLIPLoRA(fresh_base, TABULAR_FEATURE_SIZE), "MedCLIP LoRA", train_loader_clip, val_loader_clip, EPOCHS))

# Final Output
print("\n" + "="*85)
print("FINAL ASSIGNMENT COMPARISON")
print("="*85)
print(pd.DataFrame(results).set_index('Model'))


--- Training ResNet Concat (Scratch) ---


Epoch 1 | Val AUC: 0.6061 | Val F1: 0.6217 | ECE: 0.0602


Epoch 2 | Val AUC: 0.6082 | Val F1: 0.6163 | ECE: 0.1967


Epoch 3 | Val AUC: 0.6231 | Val F1: 0.6163 | ECE: 0.4313


Epoch 4 | Val AUC: 0.6870 | Val F1: 0.6312 | ECE: 0.3874


...

Epoch 6 | Val AUC: 0.6857 | Val F1: 0.6443 | ECE: 0.1318


Epoch 7 | Val AUC: 0.6904 | Val F1: 0.6557 | ECE: 0.1084


Epoch 8 | Val AUC: 0.7036 | Val F1: 0.5979 | ECE: 0.0526


Epoch 9 | Val AUC: 0.6946 | Val F1: 0.6429 | ECE: 0.0973


Epoch 10 | Val AUC: 0.6941 | Val F1: 0.6468 | ECE: 0.1209

FINAL ASSIGNMENT COMPARISON
                             Precision  Recall     AUC      F1     ECE  \
Model                                                                    
ResNet Concat (Scratch)         0.5962  1.0000  0.6870  0.6409  0.0602   
ResNet Cross-Attn (Scratch)     0.6170  1.0000  0.6613  0.6212  0.0756   
MedCLIP Linear Probe            0.4444  0.9902  0.5718  0.6121  0.0862   
MedCLIP Partial FT              0.4667  1.0000  0.6640  0.6282  0.0802   
MedCLIP LoRA                    0.6304  1.0000  0.7036  0.6557  0.0526   

                             Time (s)  GPU Mem (MB)      Params  
Model                                                            
ResNet Concat (Scratch)        201.60        913.70  11,242,433  
ResNet Cross-Attn (Scratch)    198.02        918.47  11,604,545  
MedCLIP Linear Probe           250.67        735.47         514  
MedCLIP Partial FT             244.12        736.46      65,921 